In [8]:
from langchain_core.tools import tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [62]:
@tool
def multiplication(a: int, b: int) -> int:
    """This is a function for multiply two numbers"""
    return a*b

@tool
def addition(a: int, b: int) -> int:
    """This is a function for add two numbers"""
    return a+b

In [ ]:
wiki_api = WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=50)
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api)

wiki_tool.invoke({"query":"modi"})

'a'

In [63]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-7B-Instruct',
    task='text-generation'
)

model = ChatHuggingFace(llm=llm)

# model.invoke("what is langchain")

In [37]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import HumanMessage, ToolMessage

duck_tool = DuckDuckGoSearchRun()


In [64]:
tools = [multiplication, addition, wiki_tool, duck_tool]

tools

[StructuredTool(name='multiplication', description='This is a function for multiply two numbers', args_schema=<class 'langchain_core.utils.pydantic.multiplication'>, func=<function multiplication at 0x0000017684926520>),
 StructuredTool(name='addition', description='This is a function for add two numbers', args_schema=<class 'langchain_core.utils.pydantic.addition'>, func=<function addition at 0x0000017684926480>),
 WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\Machine-Learning\\LangChain\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=5, lang='en', load_all_available_meta=False, doc_content_chars_max=50)),
 DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text'))]

In [65]:
llm_with_tools = model.bind_tools(tools)

In [28]:
query = "what is 6*9"

message = [HumanMessage(query)]
res = llm_with_tools.invoke(message)
res

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":6,"b":9}', 'name': 'multiplication', 'description': None}, 'id': 'functions.multiplication:0', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 261, 'total_tokens': 313}, 'model_name': 'moonshotai/Kimi-K2-Thinking', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--3a80aaa0-49d0-4692-9207-b4747509477b-0', tool_calls=[{'name': 'multiplication', 'args': {'a': 6, 'b': 9}, 'id': 'functions.multiplication:0', 'type': 'tool_call'}], usage_metadata={'input_tokens': 261, 'output_tokens': 52, 'total_tokens': 313})

In [29]:
res.tool_calls

[{'name': 'multiplication',
  'args': {'a': 6, 'b': 9},
  'id': 'functions.multiplication:0',
  'type': 'tool_call'}]

In [87]:
def message_for_AI(res, message):
    
    message.append(res)
    
    for i in res.tool_calls:
        
        selected = {
            "addition": addition,
            "multiplication": multiplication,
            "wikipedia": wiki_tool,
            "duckduckgo_search": duck_tool
        }[i["name"].lower()]

        tool_output = selected.invoke(i["args"])

        message.append(
            ToolMessage(
                content=str(tool_output),
                name=i["name"],
                tool_call_id=i["id"]     
            )
        )

    return llm_with_tools.invoke(message)

In [ ]:
query = "what is the status of share market in the USA and tell me about trump a bit"

message = [HumanMessage(query)]

res = llm_with_tools.invoke(message)

while hasattr(res, "tool_calls") and res.tool_calls:
    res = message_for_AI(res, message)

print(res.content)

In [84]:
from langchain_core.messages import HumanMessage
messages = []
query = "multiply 6 and 5 and then add the answer with 10"
messages = [HumanMessage(query)]

ai_msg = llm_with_tools.invoke(messages)
ai_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":6,"b":5}', 'name': 'multiplication', 'description': None}, 'id': 'call_1730f9350cgiw1antzvcws66', 'type': 'function'}, {'function': {'arguments': '{"a":30,"b":10}', 'name': 'addition', 'description': None}, 'id': 'call_093sm1zeeeibynicu96mbcxa', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 565, 'total_tokens': 619}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--0761d335-065b-49a8-bf4b-7ef9f3ad6c4b-0', tool_calls=[{'name': 'multiplication', 'args': {'a': 6, 'b': 5}, 'id': 'call_1730f9350cgiw1antzvcws66', 'type': 'tool_call'}, {'name': 'addition', 'args': {'a': 30, 'b': 10}, 'id': 'call_093sm1zeeeibynicu96mbcxa', 'type': 'tool_call'}], usage_metadata={'input_tokens': 565, 'output_tokens': 54, 'total_tokens': 619})

In [79]:
ai_msg.tool_calls

[{'name': 'duckduckgo_search',
  'args': {'query': 'recent AI news'},
  'id': 'call_cbgn2hodjlonp45ggnqzkj2y',
  'type': 'tool_call'}]

In [85]:
for tool_call in ai_msg.tool_calls:
    selected_tool = {
            "addition": addition,
            "multiplication": multiplication,
            "wikipedia": wiki_tool,
            "duckduckgo_search": duck_tool
        }[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    
messages

[HumanMessage(content='multiply 6 and 5 and then add the answer with 10', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='30', name='multiplication', tool_call_id='call_1730f9350cgiw1antzvcws66'),
 ToolMessage(content='40', name='addition', tool_call_id='call_093sm1zeeeibynicu96mbcxa')]

In [81]:
messages

[HumanMessage(content='What is langchain and what is 5*15 and summarize the recent AI news ?', additional_kwargs={}, response_metadata={}),
 ToolMessage(content="7 hours ago - Artificial Intelligence: Read latest updates on AI like Google AI, ChatGPT, Google Lamda, Bard chatbot and more along with latest news as AI technology advances and makes new progress. All get detailed articles on AI related queries like what is AI, types of artificial intelligence, its applications and future. 1 day ago - Now, researchers from Japan have developed a ... ... May 30, 2025 Students recently unveiled their invention of a robotic actuator -- the 'muscle' that converts energy into a robot's physical movement -- that has the ability to detect punctures ... October 8, 2025 - This breakthrough performance in abstract problem-solving builds on our previous gold at the International Mathematical Olympiad, proving Gemini's world-class coding and reasoning capabilities. ... Jump to position 1 Jump to positio

In [86]:
response=llm_with_tools.invoke(messages)
print(response.content)

First, let's multiply 6 and 5, which gives us 30. Then, we add 10 to this result. So, 30 + 10 equals 40. Therefore, the final answer is 40.
